In [ ]:
import pickle as pk
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Week 2 Helper Notebook
This exploration should produce several figures , which you should include in the progress report and make look nice using seaborn (feel free to use AI to help with plotting). 
1. Numerical distributions of eICU data, with a variety of different types (ie. Gaussian, skewed, etc)
2. Same variable and then apply the six encodings, for several variables. 

Also track the output dimension of each encoding!! Does it output 1 variable -> 1 variable? Or does it output M new dimensions? 


General guiding goal: 
* How do different numerical encodings transform variables with different distributional types / structures?
* What 20 features for X should we pick?


## Step 1: Load eICU data

In [ ]:
feature_sets = pk.load(open('../../data/feature_names.pkl', 'rb'))
patient_agg = pd.read_parquet('../../data/clean_dataset.parquet')

print("Dataset shape:", patient_agg.shape)
patient_agg.head()

In [ ]:
# This distinction of numerical (continuous), binary, and categorical
# columns may be useful throughout the project.
all_num_cols = (
    feature_sets['labs'].tolist()
    + feature_sets['vitals'].tolist()
    + ['age', 'admissionheight', 'admissionweight']
)

all_bin_cols = (
    feature_sets['icd10_before24h'].tolist()
    + feature_sets['admissiondx'].tolist()
)

all_cat_cols = [
    'hospital_region',
    'ethnicity',
    'gender',
    'hospital_numbedscategory',
    'hospitaldischargeyear',
    'hospitalid'
]

In [ ]:
# TODO: look at the 'age' variable. Is it continuous or categorical (hint: it is both).
# How should you handle that? 

## Step 1: Analyze + characterize numerical distributions 

In [ ]:
# TODO: characterize and plot the numerical features. You may want to create a table 
# of summary statistics for each numerical feature (eg. mean, median, max, min, quantiles, skew, n_unique...),
# and/or create histograms or boxplots to visualize their distributions.

In [ ]:
# TODO:
# Explore these stats / the above generated table.
#
# Some useful questions:
# 1. Which features have the most missingness?
# 2. Which are highly skewed?
# 3. Which have very large differences between q99 and the maximum? (= outliers)
# 4. Which features have only a small number of repeated values?
# 5. Which might be multimodal?

In [ ]:
# TODO:
# Start with ~10-15 numerical variables that appear to have different distributions.
#
# Try to include examples that seem:
#   - approximately Gaussian
#   - skewed
#   - heavy-tailed / contain outliers
#   - multimodal
#   - bounded
#   - highly concentrated / many repeated values

study_num_cols = [
    # "heart_rate",
    # "creatinine",
    # ...
]

In [ ]:
# Generate plots for the chosen numerical variables using seaborn. If there
# are outliers, then trim the x-axis only for VISUALIZATION so
#  a handful of extreme values do not
# make the whole histogram unreadable.

## Step 3: Encoding building 
All tools you build here will then be used later, so the code should be neat and productionizable. It is ideal if it fits into the following class spec: 
```
class Encoder:
    """
    Encode (preprocess) numerical variable x
    """

    def fit(self, x):
        x = np.asarray(x, dtype=float)
        OPTIONALLY DO STUFF HERE 
        return self

    def transform(self, x):
        x = np.asarray(x, dtype=float)
        DO STUFF HERE
        return ...

    def inverse_transform(self, z):
        revert z back to raw x
        ```


### Encoding 1: Standardized scalar
Hint: use sklearn or the scaler in encoders.py

In [ ]:
# todo: implement and viz on some numerical variables. 

### Encoding 2: Quantile normalization scalar
Hint: use sklearn

In [ ]:
# todo: implement and viz on some numerical variables. 

### Encoding 3: Mode specific normalization scalar
This is from the CTGAN / TVAE paper!! 


In [ ]:
from sklearn.mixture import BayesianGaussianMixture
# TODO: understand this code!!

class ModeSpecificEncoder:
    """
    Mode-specific normalization, inspired by CTGAN.

    A feature like "hours worked per week" may have several peaks ("modes"),
    e.g. one near 20 (part-time) and one near 40 (full-time). A single mean and
    std describe that badly. So we:

        1. fit a Gaussian mixture (one bell curve per peak),
        2. find the peak each value most likely belongs to,
        3. describe the value with two things:
               alpha        = (x - peak_mean) / (4 * peak_std)   position relative to its peak
               one_hot(k)   which peak it belongs to

    Output has m = 1 + K dimensions, where K is the number of peaks found.
    Example with K = 3:   x  ->  [0.17, 0, 1, 0]
    This was generated with help from Claude.
    """

    def __init__(self, max_components=10, weight_threshold=0.005, random_state=42):
        self.random_state = random_state

    def fit(self, x):
        x = np.asarray(x, dtype=float).reshape(-1, 1)

        self.model_ = BayesianGaussianMixture(
            n_components=min(self.max_components, len(np.unique(x))),
            weight_concentration_prior_type="dirichlet_process",
            weight_concentration_prior=0.001,
            max_iter=200,
            random_state=self.random_state,
        ).fit(x)

        # Keep only the peaks that a meaningful share of the data uses.
        keep = self.model_.weights_ > self.weight_threshold
        if not keep.any():
            keep[np.argmax(self.model_.weights_)] = True
        self.keep_ = np.flatnonzero(keep)

        self.means_ = self.model_.means_.reshape(-1)[self.keep_]
        variances = self.model_.covariances_.reshape(-1)[self.keep_]
        self.stds_ = np.maximum(np.sqrt(variances), 1e-6)

        self.output_dim_ = 1 + len(self.keep_)
        return self

    def transform(self, x):
        x = np.asarray(x, dtype=float).reshape(-1, 1)

        # Which peak does each value most likely belong to?
        probs = self.model_.predict_proba(x)[:, self.keep_]
        peak = probs.argmax(axis=1)

        alpha = (x[:, 0] - self.means_[peak]) / (4.0 * self.stds_[peak])
        alpha = np.clip(alpha, -0.99, 0.99)
        one_hot = np.eye(len(self.keep_))[peak]
        return np.column_stack([alpha, one_hot])

    def inverse_transform(self, z):
        z = np.asarray(z, dtype=float)
        alpha = np.clip(z[:, 0], -1.0, 1.0)
        peak = z[:, 1:].argmax(axis=1)
        return alpha * 4.0 * self.stds_[peak] + self.means_[peak]


In [ ]:
# TODO:
#
# Try this with at least:
#   1. an approximately unimodal variable
#   2. a clearly skewed variable
#   3. a variable that looks multimodal
#
# Does the Gaussian mixture identify modes that seem meaningful?
#
# Are there variables where the mixture model seems to introduce
# unnecessary modes?

### Encoding 4: Quantile binning

In [ ]:
# TODO: implement on your own and investigate how it transforms variables! 

### Encoding 5: Piecewise linear encoding (PLE)
Make sure you understand the encoding in the original paper. 


In [ ]:
import numpy as np
# TODO: understand this code!! 
 
class QuantilePLEEncoder:
    """
    Piecewise-linear encoding (PLE) with quantile bins.  Output has m = (number of bins) dimensions.
 
    Idea: cut the feature into bins that each hold about the same number of training rows,
    then describe x by "how full" each bin is.
 
    Example: bin edges [0, 10, 20, 40]  ->  3 bins (0-10, 10-20, 20-40)
 
        x =  5   ->  [0.5, 0.0, 0.0]    half way through bin 1, later bins empty
        x = 15   ->  [1.0, 0.5, 0.0]    bin 1 full, half way through bin 2
        x = 30   ->  [1.0, 1.0, 0.5]    bins 1 and 2 full, half way through bin 3
 
    Reverse direction (inverse_transform): add up how much of each bin is filled.
 
        x = lowest edge + sum over bins of (fill_fraction * bin_width)
        e.g. [1.0, 0.5, 0.0]  ->  0 + 1.0*10 + 0.5*10 + 0.0*20 = 15
    
    This was generated with help from Claude.
    """
 
    def __init__(self, n_bins=8, clip_extremes=False):
        self.n_bins = n_bins
        # False: values below the first edge / above the last edge give fill fractions
        #        outside [0, 1] (extrapolation).  True: they are squashed into [0, 1].
        self.clip_extremes = clip_extremes
 
    def fit(self, x):
        x = np.asarray(x, dtype=float).reshape(-1)
        x = x[np.isfinite(x)]
 
        # Bin edges = quantiles of the training data (0%, 12.5%, 25%, ... 100% for 8 bins).
        edges = np.quantile(x, np.linspace(0, 1, self.n_bins + 1))
        # Many repeated values can give identical edges; keep each edge once.
        self.edges_ = np.unique(edges)
 
        if len(self.edges_) < 2:
            raise ValueError("Feature has fewer than two unique quantile boundaries.")
        return self
 
    def _constrain(self, z):
        """
        Enforce the valid range of each column. Used by BOTH directions.
 
        Middle bins:  must lie in [0, 1]   (a bin cannot be less than empty or more than full)
        First bin:    at most 1            (it can go below 0 for x under the lowest edge)
        Last bin:     at least 0           (it can go above 1 for x over the highest edge)
        """
        z = z.copy()
 
        if self.clip_extremes:
            return np.clip(z, 0.0, 1.0)
 
        if z.shape[1] > 1:
            z[:, 0] = np.minimum(z[:, 0], 1.0)
            z[:, -1] = np.maximum(z[:, -1], 0.0)
            if z.shape[1] > 2:
                z[:, 1:-1] = np.clip(z[:, 1:-1], 0.0, 1.0)
        return z
 
    def transform(self, x):
        x = np.asarray(x, dtype=float).reshape(-1)
        if not np.isfinite(x).all():
            raise ValueError("transform() expects finite values.")
 
        left = self.edges_[:-1]
        widths = np.diff(self.edges_)
 
        # For every bin: how far along the bin is x?  0 = at its left edge, 1 = at its right edge.
        z = (x[:, None] - left[None, :]) / widths[None, :]
        return self._constrain(z)
 
    def inverse_transform(self, z):
        z = np.asarray(z, dtype=float)
        if z.ndim != 2 or z.shape[1] != self.output_dim_:
            raise ValueError(f"Expected an array of shape (N, {self.output_dim_}).")
 
        widths = np.diff(self.edges_)
 
        # Generated data (e.g. from a VAE decoder) can be slightly out of range or noisy,
        # so first push it back into the valid range.
        z = self._constrain(z)
 
        # Start at the lowest edge and add the filled part of every bin.
        return self.edges_[0] + (z * widths[None, :]).sum(axis=1)
 
    def fit_transform(self, x):
        return self.fit(x).transform(x)
 

In [ ]:
# TODO: what does the output look like for one feature? 

### Encoding 6: Your choice 
One idea is periodic encoding. This idea is a bit more tricky to implement because it requires learnable features, so the Encoder method will contain torch modules that are parameters for the VAE. 
You can use https://github.com/yandex-research/rtdl-num-embeddings/blob/main/package/README.md if you wise. 

Or peruse literature / brainstorm for another encoding / preprocessing idea! Just motivate why you think this is a good idea for some set of numerical distributions. 

In [ ]:
# TODO: implement and inspect. What is it actually doing? 

## Step 4: Compare the six encodings

In [ ]:
## TODO: compare a numerical variable's encoded values for all the above encodings , for 
# all types of variables 

# Then
# Discuss with your group which are best / the pros and cons of each appraoch. 
# For example, you can consider
# Which methods are robust to large outliers?
# Which methods explicitly represent multimodality?
# Which methods greatly increase the input dimensionality?
# ...


## Step 5: Define the features and splits 

In [ ]:
# TODO: Finalize your features X you are learning a representation for. 
# This should be your chosen numerical variables from above, plus several other 
# categorical / binary variables. Make sure you have some set of outcomes Y
# that are not in X for which you can evaluate the quality of your representation. 

# X should contain around ~20 features.

In [ ]:
# TODO: fix a seed that freezes / makes deterministic what the train / val / test split is
# ie use train_test_split from sklearn twice, ie first for trainval / test, and then for train /val 

In [ ]:
# TODO: train a simple classifier on X to predict some outcome Y not in X. What is the 
# baseline performance on the val data?  
# Impute Nan values with a SimpleImputer. We'll revisit this next week!! 

In [ ]:
# IMPORTANT RULE FOR ALL FUTURE EXPERIMENTS
# Anything that "learns" something about a feature distribution
# MUST be fit using TRAINING DATA ONLY.
# Examples:
#   Standardization:
#       mean and std come from train_df
#   Mode specific norm:
#       Gaussian mixture is fit on train_df
# And then val_df and test_df should only be TRANSFORMED using parameters
# learned from train_df.

In [ ]:
# TODO: what if you apply any of the 6 encodings to X (using training data)? Does the classifier improve? 

In [ ]:
# TODO: copy all the "encoding" functions to a new python file. Then you can load them in
# for future experiments. For example, if you move them to a file called "encoders.py", then 
# you can load them with: `from encoders import ModeSpecificNormalizer` for example!